# 🚀 ULTIMATE TRAINING & DEPLOYMENT SYSTEM - TEKNOFEST 2025\n\nComplete pipeline: Train → Save to Drive → Deploy with ngrok

In [ ]:
# Mount Google Drive FIRST - CRITICAL!\nfrom google.colab import drive\ndrive.mount('/content/drive')\n\nimport os\nDRIVE_PATH = '/content/drive/MyDrive/TEKNOFEST_2025'\nos.makedirs(DRIVE_PATH, exist_ok=True)\nos.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)\nos.makedirs(f'{DRIVE_PATH}/final_model', exist_ok=True)\nprint(f'✅ Drive mounted! Save path: {DRIVE_PATH}')

In [ ]:
# Install dependencies\n!pip install unsloth transformers trl datasets accelerate -q\n!pip install flask flask-cors pyngrok -q\n\n# For Unsloth\nimport torch\nmajor_version, minor_version = torch.cuda.get_device_capability()\n!pip install \"unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git\" -q\n\nif major_version >= 8:\n    !pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes -q\nelse:\n    !pip install --no-deps xformers trl peft accelerate bitsandbytes -q\n\nprint('✅ All dependencies installed!')

In [ ]:
from unsloth import FastLanguageModel\nimport torch\nfrom datetime import datetime\nimport json\nimport shutil\n\n# Model configuration\nmax_seq_length = 2048\ndtype = None\nload_in_4bit = True\n\n# Load base model\nprint('🔥 Loading Gemma 3N E4B model...')\nmodel, tokenizer = FastLanguageModel.from_pretrained(\n    model_name='unsloth/gemma-2-2b-it-bnb-4bit',\n    max_seq_length=max_seq_length,\n    dtype=dtype,\n    load_in_4bit=load_in_4bit,\n)\n\n# Add LoRA adapters\nmodel = FastLanguageModel.get_peft_model(\n    model,\n    r=16,\n    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',\n                    'gate_proj', 'up_proj', 'down_proj'],\n    lora_alpha=16,\n    lora_dropout=0,\n    bias='none',\n    use_gradient_checkpointing='unsloth',\n    random_state=42,\n)\n\nprint(f'✅ Model loaded with {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters')

In [ ]:
# Load your dataset\nfrom datasets import Dataset\nimport pandas as pd\n\n# Option 1: Upload your JSONL file\n# from google.colab import files\n# uploaded = files.upload()\n# dataset_file = list(uploaded.keys())[0]\n\n# Option 2: Load from Drive\n# dataset_file = '/content/drive/MyDrive/gemma3n_rich_metadata_training.jsonl'\n\n# Option 3: Create sample dataset for testing\nsample_data = [\n    {\"text\": \"<start_of_turn>user\\n<emotion>angry</emotion>\\nFaturamı öğrenmek istiyorum!<end_of_turn>\\n<start_of_turn>assistant\\nSayın müşterimiz, yaşadığınız sorun için özür dileriz. Hemen faturanızı kontrol ediyorum. [get_current_balance] Güncel bakiyeniz 250 TL görünüyor.<end_of_turn>\"},\n    {\"text\": \"<start_of_turn>user\\n<emotion>confused</emotion>\\neSIM nasıl aktive edilir?<end_of_turn>\\n<start_of_turn>assistant\\nTabii, size adım adım anlatayım. [check_device_compatibility] Önce cihazınızın uyumlu olduğunu kontrol edelim.<end_of_turn>\"},\n]\n\n# Create dataset\ndataset = Dataset.from_list(sample_data)\nprint(f'✅ Dataset loaded with {len(dataset)} examples')

In [ ]:
# INTELLIGENT TRAINING CONFIGURATION WITH AGGRESSIVE SAVING\nfrom trl import SFTTrainer\nfrom transformers import TrainingArguments, TrainerCallback\nfrom unsloth import is_bfloat16_supported\nimport numpy as np\n\nclass SaveToDriveCallback(TrainerCallback):\n    \"\"\"Save EVERY checkpoint to Google Drive\"\"\"\n    \n    def __init__(self, drive_path):\n        self.drive_path = drive_path\n        self.checkpoints_saved = []\n    \n    def on_save(self, args, state, control, **kwargs):\n        \"\"\"Copy checkpoint to Drive after each save\"\"\"\n        checkpoint_name = f\"checkpoint-{state.global_step}\"\n        source = os.path.join(args.output_dir, checkpoint_name)\n        destination = os.path.join(self.drive_path, 'checkpoints', checkpoint_name)\n        \n        if os.path.exists(source):\n            print(f\"\\n💾 Copying checkpoint to Drive: {checkpoint_name}\")\n            shutil.copytree(source, destination, dirs_exist_ok=True)\n            self.checkpoints_saved.append(checkpoint_name)\n            print(f\"✅ Saved to Drive: {destination}\")\n\nclass IntelligentLRScheduler(TrainerCallback):\n    \"\"\"Dynamic LR adjustment based on loss patterns\"\"\"\n    \n    def __init__(self):\n        self.loss_history = []\n        self.lr_adjustments = []\n        self.best_loss = float('inf')\n        self.patience = 0\n    \n    def on_log(self, args, state, control, logs=None, **kwargs):\n        if logs and 'loss' in logs:\n            current_loss = logs['loss']\n            self.loss_history.append(current_loss)\n            \n            # Smart LR adjustment\n            if len(self.loss_history) > 5:\n                recent_losses = self.loss_history[-5:]\n                \n                # If loss stopped improving\n                if np.std(recent_losses) < 0.01:\n                    self.patience += 1\n                    if self.patience > 3:\n                        # Reduce LR\n                        current_lr = state.log_history[-1].get('learning_rate', 5e-5)\n                        new_lr = max(current_lr * 0.5, 1e-6)\n                        for param_group in kwargs['model'].optimizer.param_groups:\n                            param_group['lr'] = new_lr\n                        print(f\"\\n🔄 LR reduced: {current_lr:.2e} → {new_lr:.2e}\")\n                        self.patience = 0\n                else:\n                    self.patience = 0\n            \n            # Print smart status\n            if state.global_step % 5 == 0:\n                trend = \"📉\" if current_loss < self.best_loss else \"📈\"\n                self.best_loss = min(self.best_loss, current_loss)\n                print(f\"Step {state.global_step}: Loss {current_loss:.4f} {trend} | Best: {self.best_loss:.4f}\")\n\n# Create trainer with aggressive saving\ntraining_args = TrainingArguments(\n    per_device_train_batch_size=1,\n    gradient_accumulation_steps=16,\n    warmup_steps=10,\n    max_steps=100,  # Adjust based on your needs\n    learning_rate=5e-5,\n    fp16=not is_bfloat16_supported(),\n    bf16=is_bfloat16_supported(),\n    logging_steps=1,\n    optim=\"adamw_8bit\",\n    weight_decay=0.01,\n    lr_scheduler_type=\"cosine\",\n    seed=42,\n    output_dir=\"./outputs\",\n    \n    # AGGRESSIVE SAVING STRATEGY\n    save_strategy=\"steps\",\n    save_steps=5,  # Save every 5 steps\n    save_total_limit=None,  # Keep ALL checkpoints\n    save_safetensors=True,\n    \n    # Load best model at end\n    load_best_model_at_end=True,\n    metric_for_best_model=\"loss\",\n    greater_is_better=False,\n    \n    remove_unused_columns=False,\n    report_to=\"none\",\n)\n\n# Initialize callbacks\nsave_callback = SaveToDriveCallback(DRIVE_PATH)\nlr_scheduler = IntelligentLRScheduler()\n\ntrainer = SFTTrainer(\n    model=model,\n    tokenizer=tokenizer,\n    train_dataset=dataset,\n    dataset_text_field=\"text\",\n    max_seq_length=max_seq_length,\n    dataset_num_proc=2,\n    packing=False,\n    args=training_args,\n    callbacks=[save_callback, lr_scheduler],\n)\n\nprint(\"✅ Trainer configured with intelligent features and Drive saving!\")

In [ ]:
# TRAIN WITH AUTOMATIC SAVING TO DRIVE\nimport time\n\nprint(\"\"\"\n╔════════════════════════════════════════════════════════════╗\n║  🚀 STARTING TRAINING WITH DRIVE BACKUP                    ║\n╠════════════════════════════════════════════════════════════╣\n║  • Checkpoints saved every 5 steps                         ║\n║  • All checkpoints backed up to Drive                      ║\n║  • Intelligent LR adjustment                               ║\n║  • Best model will be saved at end                         ║\n╚════════════════════════════════════════════════════════════╝\n\"\"\")\n\nstart_time = time.time()\n\ntry:\n    # Train the model\n    trainer.train()\n    \n    training_time = time.time() - start_time\n    print(f\"\\n✅ Training completed in {training_time/60:.1f} minutes!\")\n    \n    # Save final model to Drive\n    print(\"\\n💾 Saving final model to Drive...\")\n    final_path = os.path.join(DRIVE_PATH, 'final_model')\n    model.save_pretrained(final_path)\n    tokenizer.save_pretrained(final_path)\n    \n    # Save training info\n    training_info = {\n        'training_time_minutes': training_time/60,\n        'final_loss': trainer.state.log_history[-1].get('loss', 'N/A'),\n        'total_steps': trainer.state.global_step,\n        'checkpoints_saved': save_callback.checkpoints_saved,\n        'timestamp': datetime.now().isoformat()\n    }\n    \n    with open(f'{final_path}/training_info.json', 'w') as f:\n        json.dump(training_info, f, indent=2)\n    \n    print(f\"✅ Final model saved to: {final_path}\")\n    print(f\"✅ Checkpoints saved: {len(save_callback.checkpoints_saved)}\")\n    \nexcept Exception as e:\n    print(f\"❌ Training error: {e}\")\n    print(\"💾 Attempting to save emergency checkpoint...\")\n    emergency_path = os.path.join(DRIVE_PATH, 'emergency_checkpoint')\n    model.save_pretrained(emergency_path)\n    tokenizer.save_pretrained(emergency_path)\n    print(f\"✅ Emergency checkpoint saved to: {emergency_path}\")\n    raise

## 🚀 DEPLOYMENT - Serve Model via ngrok

In [ ]:
# Prepare model for inference\nFastLanguageModel.for_inference(model)\nprint(\"✅ Model prepared for inference\")

In [ ]:
# Create Flask server for deployment\nfrom flask import Flask, request, jsonify\nfrom flask_cors import CORS\nfrom pyngrok import ngrok\nimport base64\nimport numpy as np\n\napp = Flask(__name__)\nCORS(app)\n\n@app.route('/health', methods=['GET'])\ndef health():\n    return jsonify({\n        \"status\": \"healthy\",\n        \"model\": \"gemma3n-teknofest-finetuned\",\n        \"device\": str(torch.cuda.get_device_name(0)),\n        \"checkpoint\": \"final_model\",\n        \"ready\": True\n    })\n\n@app.route('/predict', methods=['POST'])\ndef predict():\n    try:\n        data = request.json\n        emotion = data.get('emotion', 'neutral')\n        text = data.get('text', 'Merhaba')\n        \n        # Create prompt\n        prompt = f\"\"\"<start_of_turn>user\n<emotion>{emotion}</emotion>\n{text}\n<end_of_turn>\n<start_of_turn>assistant\"\"\"\n        \n        # Generate\n        inputs = tokenizer(prompt, return_tensors=\"pt\").to(model.device)\n        \n        with torch.no_grad():\n            outputs = model.generate(\n                **inputs,\n                max_new_tokens=200,\n                temperature=0.7,\n                do_sample=True,\n                top_p=0.95,\n            )\n        \n        response = tokenizer.decode(outputs[0], skip_special_tokens=True)\n        response = response.split(\"assistant\")[-1].strip()\n        \n        # Extract tools from response\n        tools = []\n        if \"[\" in response and \"]\" in response:\n            import re\n            tools = re.findall(r'\\[([^\\]]+)\\]', response)\n        \n        return jsonify({\n            \"generated_text\": response,\n            \"emotion_detected\": emotion,\n            \"tools\": tools,\n            \"model\": \"finetuned\",\n            \"success\": True\n        })\n        \n    except Exception as e:\n        return jsonify({\"error\": str(e)}), 500\n\n@app.route('/model_info', methods=['GET'])\ndef model_info():\n    return jsonify({\n        \"model_path\": DRIVE_PATH + '/final_model',\n        \"parameters\": sum(p.numel() for p in model.parameters()),\n        \"trainable_parameters\": sum(p.numel() for p in model.parameters() if p.requires_grad),\n        \"checkpoints_available\": save_callback.checkpoints_saved if 'save_callback' in globals() else [],\n        \"training_completed\": True\n    })\n\nprint(\"✅ Flask server created\")

In [ ]:
# Start ngrok tunnel\npublic_url = ngrok.connect(5000)\n\nprint(\"\"\"\n╔════════════════════════════════════════════════════════════╗\n║  🎉 MODEL DEPLOYED SUCCESSFULLY!                           ║\n╠════════════════════════════════════════════════════════════╣\n║  Your finetuned model is now accessible via:               ║\n\"\"\")\nprint(f\"║  {public_url}                                             ║\")\nprint(\"\"\"╠════════════════════════════════════════════════════════════╣\n║  Test endpoints:                                           ║\n║  • /health - Check server status                           ║\n║  • /predict - Get model predictions                        ║\n║  • /model_info - Model information                         ║\n╚════════════════════════════════════════════════════════════╝\n\"\"\")\n\nprint(f\"\\n📋 Copy this URL to use in your local system: {public_url}\")\nprint(\"\\n⚠️ Keep this cell running! Don't stop it.\")

In [ ]:
# Run the server (THIS WILL BLOCK - Keep running!)\nprint(\"🚀 Server starting... (Keep this running!)\")\napp.run(port=5000, debug=False)

## 📁 Verify Drive Saves\n\nCheck your saved models:

In [ ]:
# List all saved files in Drive\nimport os\n\nprint(\"📁 Files saved to Google Drive:\")\nprint(\"=\" * 50)\n\n# Check final model\nfinal_model_path = f\"{DRIVE_PATH}/final_model\"\nif os.path.exists(final_model_path):\n    files = os.listdir(final_model_path)\n    print(f\"\\n✅ Final Model ({len(files)} files):\")\n    for f in files[:5]:\n        size = os.path.getsize(f\"{final_model_path}/{f}\") / (1024*1024)\n        print(f\"   - {f} ({size:.1f} MB)\")\n\n# Check checkpoints\ncheckpoints_path = f\"{DRIVE_PATH}/checkpoints\"\nif os.path.exists(checkpoints_path):\n    checkpoints = os.listdir(checkpoints_path)\n    print(f\"\\n✅ Checkpoints ({len(checkpoints)} saved):\")\n    for cp in checkpoints:\n        print(f\"   - {cp}\")\n\n# Training info\ninfo_path = f\"{final_model_path}/training_info.json\"\nif os.path.exists(info_path):\n    with open(info_path, 'r') as f:\n        info = json.load(f)\n    print(f\"\\n📊 Training Info:\")\n    print(f\"   - Duration: {info['training_time_minutes']:.1f} minutes\")\n    print(f\"   - Final Loss: {info['final_loss']}\")\n    print(f\"   - Total Steps: {info['total_steps']}\")\n    print(f\"   - Timestamp: {info['timestamp']}\")\n\nprint(\"\\n✅ All models successfully saved to Google Drive!\")

## 🔄 Load Model from Drive (For Later Use)

In [ ]:
# Load your saved model from Drive\nfrom unsloth import FastLanguageModel\n\n# Path to your saved model\nsaved_model_path = f\"{DRIVE_PATH}/final_model\"\n\nprint(f\"Loading model from: {saved_model_path}\")\n\n# Load the finetuned model\nmodel, tokenizer = FastLanguageModel.from_pretrained(\n    model_name=saved_model_path,\n    max_seq_length=2048,\n    dtype=None,\n    load_in_4bit=True,\n)\n\nFastLanguageModel.for_inference(model)\n\nprint(\"✅ Model loaded from Drive successfully!\")